In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/data.yaml
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/styro_000203.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/recycle_001390.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/recycle_000129.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/organic_000146.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/recycle_001308.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/recycle_001414.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/recycle_000842.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/styro_000217.txt
/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset/valid/labels/styro_000052.txt
/kaggle/

## 1. Setup Environment

In [2]:
# Fix NumPy 2.x compatibility FIRST
!pip install ultralytics --quiet
!pip install "numpy<2" --quiet

print(" Ultralytics installed successfully!")
print(" NumPy downgraded to 1.x for compatibility!")
print("  Using Kaggle pre-installed packages (torch, cv2, etc.)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
import os
import gc
import torch
import numpy as np
import pandas as pd
import yaml
import shutil
from pathlib import Path
from ultralytics import YOLO

# Verify GPU
print("="*60)
print(" SYSTEM INFORMATION")
print("="*60)
print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    
    # Recommend batch size based on GPU memory
    if gpu_mem >= 15:  # P100 16GB
        recommended_batch = 16
    elif gpu_mem >= 10:
        recommended_batch = 12
    else:
        recommended_batch = 8
    print(f"\n Recommended batch size: {recommended_batch}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
 SYSTEM INFORMATION
PyTorch version: 2.6.0+cu124
NumPy version: 1.26.4
CUDA available: True
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB

 Recommended batch size: 16


In [4]:
# ============================================
# CẤU HÌNH ĐƯỜNG DẪN - FIX CỨNG CHO KAGGLE
# ============================================

DATASET_PATH = "/kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset"
OUTPUT_DIR = "/kaggle/working"

# Verify dataset exists
print("="*60)
print(" DATASET VERIFICATION")
print("="*60)

if os.path.exists(DATASET_PATH):
    print(f" Dataset found at: {DATASET_PATH}")
    
    # Count images in each split
    for split in ['train', 'valid', 'test']:
        img_path = os.path.join(DATASET_PATH, split, 'images')
        if os.path.exists(img_path):
            count = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.jpeg', '.png'))])
            print(f"   {split}: {count} images")
else:
    print(f" Dataset NOT found at: {DATASET_PATH}")
    print("Please check the dataset path!")

 DATASET VERIFICATION
 Dataset found at: /kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset
   train: 13447 images
   valid: 2181 images
   test: 1018 images


## 2. Dataset Configuration

In [5]:
# 40 waste classes - cập nhật đúng với dataset
CLASS_NAMES = {
    # Organic (0-32) - 33 classes
    0: 'Apple', 1: 'Apple-core', 2: 'Apple-peel', 3: 'Bone', 4: 'Bone-fish',
    5: 'Bread', 6: 'Bun', 7: 'Egg', 8: 'Egg-hard', 9: 'Egg-scramble',
    10: 'Egg-shell', 11: 'Egg-steam', 12: 'Egg-yolk', 13: 'Fish', 14: 'Meat',
    15: 'Mussel', 16: 'Mussel-shell', 17: 'Noodle', 18: 'Orange', 19: 'Orange-peel',
    20: 'Other-waste', 21: 'Pancake', 22: 'Pasta', 23: 'Pear', 24: 'Pear-core',
    25: 'Pear-peel', 26: 'Potato', 27: 'Rice', 28: 'Shrimp', 29: 'Shrimp-shell',
    30: 'Tofu', 31: 'Tomato', 32: 'Vegetable',
    # Inorganic (33-34) - 2 classes
    33: 'plastic_bag', 34: 'styrofoam',
    # Recyclable (35-39) - 5 classes
    35: 'Cardboard', 36: 'Glass', 37: 'Metal', 38: 'Paper', 39: 'Plastic'
}

# Convert dict to list for YOLO
CLASS_NAMES_LIST = [CLASS_NAMES[i] for i in range(len(CLASS_NAMES))]

print(f"\n CLASS DISTRIBUTION:")
print(f"   Total classes: {len(CLASS_NAMES)}")
print(f"   Organic (0-32): 33 classes")
print(f"   Inorganic (33-34): 2 classes")
print(f"   Recyclable (35-39): 5 classes")


 CLASS DISTRIBUTION:
   Total classes: 40
   Organic (0-32): 33 classes
   Inorganic (33-34): 2 classes
   Recyclable (35-39): 5 classes


In [6]:
# Create data.yaml for training
data_yaml = {
    'path': DATASET_PATH,
    'train': 'train/images',
    'val': 'valid/images', 
    'test': 'test/images',
    'nc': len(CLASS_NAMES),
    'names': CLASS_NAMES_LIST
}

yaml_path = f"{OUTPUT_DIR}/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f" Created data.yaml at: {yaml_path}")

# Verify yaml content
print("\n data.yaml content:")
with open(yaml_path, 'r') as f:
    print(f.read()[:500] + "...")

 Created data.yaml at: /kaggle/working/data.yaml

 data.yaml content:
names:
- Apple
- Apple-core
- Apple-peel
- Bone
- Bone-fish
- Bread
- Bun
- Egg
- Egg-hard
- Egg-scramble
- Egg-shell
- Egg-steam
- Egg-yolk
- Fish
- Meat
- Mussel
- Mussel-shell
- Noodle
- Orange
- Orange-peel
- Other-waste
- Pancake
- Pasta
- Pear
- Pear-core
- Pear-peel
- Potato
- Rice
- Shrimp
- Shrimp-shell
- Tofu
- Tomato
- Vegetable
- plastic_bag
- styrofoam
- Cardboard
- Glass
- Metal
- Paper
- Plastic
nc: 40
path: /kaggle/input/waste-organic-inorganic-reclycable-yolov8/Final_dataset
tes...


## 3. Utility Functions

In [7]:
def clear_memory():
    """Giải phóng GPU và RAM memory"""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
    
    # Print current memory status
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        free = (torch.cuda.get_device_properties(0).total_memory / 1e9) - reserved
        print(f" Memory cleared! Free: {free:.2f}GB")

# Clear memory before training
clear_memory()

 Memory cleared! Free: 17.06GB


## 4.  Train YOLOv8s - Balanced Speed & Accuracy

In [8]:
print("="*60)
print(" TRAINING YOLOv8s - BALANCED CONFIGURATION")
print("="*60)
print("\n Training Configuration:")
print("   Model: YOLOv8s (Small - Balance Speed & Accuracy)")
print("   Epochs: 100")
print("   Batch size: 16")
print("   Image size: 640")
print("   Optimizer: AdamW")
print("   Learning rate: 0.001 → 0.0001 (cosine)")
print("   Augmentation: Strong (mosaic, mixup, hsv, flip)")
print("\n YOLOv8s Advantages:")
print("    2x faster than YOLO11n")
print("    Better accuracy than YOLOv8n")
print("    Proven stable architecture")
print("    11.2M params (vs 6.2M YOLOv8n, 9.4M YOLO11n)")
print("="*60)

# Load pretrained YOLOv8s
model = YOLO('yolov8s.pt')
results = model.train(
    data=yaml_path,
    
    # === Training Duration ===
    epochs=100,
    patience=20,
    
    # === Batch & Hardware ===
    batch=16,
    imgsz=640,
    device=0,
    workers=4,
    
    # === Optimizer ===
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    
    # === Warmup ===
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # === Loss Weights (UPDATED) ===
    box=7.5,
    cls=1.5,              
    dfl=1.5,
    
    # === Augmentation (UPDATED) ===
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,          
    copy_paste=0.15,      
    
    # === Memory ===
    cache=False,
    amp=True,
    
    # === Output ===
    project=f"{OUTPUT_DIR}/runs",
    name="yolov8s_waste",
    exist_ok=True,
    pretrained=True,
    save=True,
    save_period=20,
    plots=True,
    val=True,
    
    # === Advanced (UPDATED) ===
    cos_lr=True,
    close_mosaic=10,
    label_smoothing=0.05,
    nbs=64,
    overlap_mask=True,
    mask_ratio=4,
    seed=42,
)

print("\n" + "="*60)
print(" TRAINING COMPLETED!")
print("="*60)

 TRAINING YOLOv8s - BALANCED CONFIGURATION

 Training Configuration:
   Model: YOLOv8s (Small - Balance Speed & Accuracy)
   Epochs: 100
   Batch size: 16
   Image size: 640
   Optimizer: AdamW
   Learning rate: 0.001 → 0.0001 (cosine)
   Augmentation: Strong (mosaic, mixup, hsv, flip)

 YOLOv8s Advantages:
    2x faster than YOLO11n
    Better accuracy than YOLOv8n
    Proven stable architecture
    11.2M params (vs 6.2M YOLOv8n, 9.4M YOLO11n)
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.3.235 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.15, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, d

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all       2181       3252      0.748      0.658      0.706      0.474
                 Apple         12         37      0.736      0.568      0.652      0.514
            Apple-core         26         31      0.855      0.613      0.774      0.431
            Apple-peel         33         79      0.527      0.608      0.589      0.321
                  Bone         81        121      0.754      0.633       0.74      0.358
             Bone-fish         20         21      0.625      0.524      0.657      0.377
                 Bread         21         36       0.88      0.889      0.904      0.576
                   Bun          1          1      0.602          1      0.995      0.895
                   Egg          1          1      0.351          1      0.995      0.995
              Egg-hard          1          2          1          0          0          0
          Egg-scramble         57        141      0.803      0.816       0.88      0.557
             Egg-shel